In [ ]:
import re
import os
import pandas as pd
import numpy as np
from pathlib import PurePath, PurePosixPath

import sys
from h_anonypy.modules_dicom import get_folder_list, replace_digits, check_dicom_from_folder
from h_anonypy.modules_dicom import get_patient_info, anonymize_dicom_file
from h_anonypy.modules_dicom import get_representative_files_from_dicom_folders
from medcore import ImageReader
import pydicom
from pydicom.errors import InvalidDicomError

def get_unique_uid_filepaths(root_dir, uid_key="SeriesInstanceUID"):
    seen_uids = set()
    representative_files = []

    for dirpath, _, filenames in os.walk(root_dir):
        for fname in filenames:
            if fname == "DICOMDIR" or fname.startswith("._"):
                continue

            filepath = os.path.join(dirpath, fname)

            try:
                ds = pydicom.dcmread(
                    filepath,
                    stop_before_pixels=True,
                    specific_tags=[uid_key],
                )
            except InvalidDicomError:
                continue
            except Exception:
                continue

            uid_value = ds.get(uid_key)
            if not uid_value:
                continue

            uid_value = str(uid_value)

            if uid_value not in seen_uids:
                seen_uids.add(uid_value)
                representative_files.append(filepath)

    return representative_files


def extract_uids_from_folder(root_dir, uid_key="SeriesInstanceUID"):
    uid_list = []

    for dirpath, _, filenames in os.walk(root_dir):
        for fname in filenames:
            if fname == "DICOMDIR" or fname.startswith("._"):
                continue

            filepath = os.path.join(dirpath, fname)

            try:
                ds = pydicom.dcmread(
                    filepath,
                    stop_before_pixels=True,
                    specific_tags=[uid_key],
                )
                uid_value = ds.get(uid_key)
                if uid_value:
                    uid_list.append(str(uid_value))
            except InvalidDicomError:
                continue
            except Exception:
                continue

    return uid_list


def get_dicom_from_subfolder(root_dir):
    candidate_folders = []

    for dirpath, _, filenames in os.walk(root_dir):
        valid_files = [
            os.path.join(dirpath, fname)
            for fname in filenames
            if fname != "DICOMDIR" and not fname.startswith("._")
        ]
        if valid_files:
            candidate_folders.append((dirpath, len(valid_files), valid_files))

    if not candidate_folders:
        return False

    candidate_folders.sort(key=lambda x: x[1], reverse=True)

    for _, _, files in candidate_folders:
        for filepath in files:
            try:
                pydicom.dcmread(
                    filepath,
                    stop_before_pixels=True,
                    specific_tags=["SOPInstanceUID"],
                )
                return filepath
            except InvalidDicomError:
                continue
            except Exception:
                continue

    return False


def get_meta_info(dicom_file):
    ds = pydicom.dcmread(dicom_file)
    ds.SpecificCharacterSet = 'ISO_IR 192'  # UTF-8
    # ds.SpecificCharacterSet = 'ISO_IR 149'  # EUC-KR
    ds.decode()
    info = {
        'PatientID':    ds.get('PatientID'),
        'PatientName':  str(ds.get('PatientName')),
        'PatientSex':   ds.get('PatientSex'),
        'PatientAge':   ds.get('PatientAge'),
        'PatientBirthDate': ds.get('PatientBirthDate'),
        'AcquisitionDate': ds.get('AcquisitionDate', 'No AcquisitionDate'),
        'PatientSize': ds.get('PatientSize'),
        'PatientWeight': ds.get('PatientWeight'),
        'OtherPatientIDs': ds.get('OtherPatientIDs'),
        'OtherPatientNames': str(ds.get('OtherPatientNames')),
        'InstitutionName': ds.get('InstitutionName'),
        'ReferringPhysicianName': str(ds.get('ReferringPhysicianName')),
        'AccessionNumber': ds.get('AccessionNumber'),
        'Modality': ds.get('Modality'),
        'BodyPartExamined': ds.get('BodyPartExamined'),
        'Manufacturer': str(ds.get('Manufacturer')),
        'ManufacturerModelName': str(ds.get('ManufacturerModelName'))

    }
    return info



In [ ]:
# check spacing for nift files

from tqdm.notebook import tqdm
from pathlib import Path

sys.path.append('/home/yhchoi/PROJECT/Pneumo')
from pneumo_reconpy import (ImageReader)


# IMAGE_META = pd.read_excel('./SHEET/IMAGE_META.xlsx', sheet_name=None)
# VIDEO_META = pd.read_excel('./SHEET/VIDEO_META.xlsx', sheet_name=None)
# HUTOM_ID = pd.read_excel('./SHEET/HUTOM_ID.xlsx', sheet_name=None)

UGI_META = IMAGE_META['UGI'].copy()
UGI_META_CT = UGI_META[UGI_META['Modality']=='CT'].copy()
UGI_META_CT = UGI_META_CT[~(UGI_META_CT['Center']=='GradientHealth')].reset_index(drop=True)

select = UGI_META_CT[UGI_META_CT['AcquisitionDate']=='19000101'].reset_index(drop=True)
select['N.slices'] = 0
select['Thickness'] = 0

hid_list = select['hutom_id'].tolist()
base_dir = Path('/nas/nas6/UGI/CT_SurgGram')
for i in tqdm(range(len(hid_list))):

    sample_folder = base_dir / hid_list[i] / '00_DICOM'

    sample_volumes = []
    sample_thcknesses = []
    sample_paths = []
    for dirpath, _, filenames in os.walk(sample_folder):
        for filename in filenames:
            if '_iso.' not in filename:
                sample_dir = Path(dirpath) / filename
                sample_volume = ImageReader(sample_dir).sitk_volume
                sample_volumes.append(sample_volume)
                sample_thcknesses.append(sample_volume.GetSpacing()[2])
                sample_paths.append(Path(dirpath))

    # slicethickness가 가장 얇은 볼륨 선택
    idx = np.argmin(sample_thcknesses)
    sample_dir = sample_paths[idx]
    sample_volume = sample_volumes[idx]
    select.loc[i,'N.slices'] = sample_volume.GetDepth()
    select.loc[i,'Thickness'] = sample_thcknesses[idx]

select



In [ ]:
##### INPUT #####
# IMAGE_META : 전체 영상 메타정보 
# HUTOM_ID : 전체 hutom id 리스트
# dicom_dir : 반입 데이터 경로
# center: 반입 기관
# importdate: 반입 날짜
# organ: 조직 정보
# n_digits: hutom id 자리수

base_dir = '/nas/nas6/DataTeam/'
dicom_dir = [base_dir+'LIVER/20250811_Liver_CT[CT_MRI]신촌세브란스병원-한대훈-250730_NN건/Liver_DICOM_0730_6_pending']
center = ['TEST']
importdate = ['20260000']
organ = ['LIVER']
n_digits = 4

IMAGE_META = pd.read_excel('/disk1/users/da_cyh_0/PROJECT/DATA/anony/DB/IMAGE_META.xlsx')
hids_all = pd.read_excel('/disk1/users/da_cyh_0/PROJECT/DATA/anony/DB/HID_ALL.xlsx')


In [ ]:
dicom_dir = [base_dir+'LIVER/20250731_[CT_MRI]신촌세브란스병원-한대훈-250730_NN건/[CT_MRI]신촌세브란스병원-한대훈-250730_NN건/Liver_Dicom']

i = 0

# 0. 초기 설정
save_dir = os.path.join(dicom_dir[i], 'ANONYMOUS')

HUTOM_ID = hids_all[hids_all['hutom_id'].astype(str).str.contains(organ[i], na=False)]
HUTOM_ID = HUTOM_ID[~HUTOM_ID['hutom_id'].astype(str).str.contains('FDA', na=False)]
HUTOM_ID = HUTOM_ID.sort_values('hutom_id')
hids = HUTOM_ID['hutom_id'].tolist()
nums = [int(m.group(1)) for s in hids if (m := re.search(r'(\d+)$', s))]
id_number = np.sort(nums)[-1] + 1

# 1. 입력 폴더 내 샘플 폴더 리스트
sample_list = get_folder_list(dicom_dir[i])

# 2. 샘플 기준, 메타 정보 추출
sample_info_list = []
for j in tqdm(range(len(sample_list))):

    # 2.1. Unique UID 기준, DICOM 리스트
    sample_path = os.path.join(dicom_dir[i], sample_list[j])
    dcmfiles = get_representative_files_from_dicom_folders(sample_path)
    # 2.2. DICOM 헤더 추출
    for dcmfile in dcmfiles:
        metadata = get_patient_info(dcmfile)
        fpath = os.path.dirname(dcmfile)
        fpath = fpath.replace(os.path.join(base_dir,organ[i]), "")
        metadata['folder'] = fpath[1:]

        sample_info_list.append(metadata)
    
# 3. 모든 데이터 기준, 메타 정보
sample_info = pd.DataFrame.from_dict(sample_info_list)
sample_info['Center'] = center[i]
sample_info['ImportDate'] = importdate[i]
sample_info.insert(0, 'hutom_id', None)

# 4. 중복 체크 및 ID 부여
sids = sample_info['PatientID'].unique().tolist()
for k in range(len(sids)):
    sample = sample_info[sample_info['PatientID']==sids[k]]
    sub_meta = IMAGE_META[IMAGE_META['series_instance_uid'].isin(sample['SeriesInstanceUID'].tolist())]

    if len(sub_meta) > 0:
        sample_info.loc[sample.index, 'hutom_id'] = sub_meta['hutom_id'].unique()[0]
    else:
        hutomid = f"{organ[i]}{id_number:0{n_digits}d}"
        sample_info.loc[sample.index, 'hutom_id'] = hutomid
        id_number += 1

# 5. 익명화
for k in range(len(sids)):
    sample = sample_info[sample_info['PatientID']==sids[k]]

    sample_path = sample["folder"].tolist()
    hutomid = sample['hutom_id'].tolist()    
    # 5.1 하위 폴더 기준, 모든 DICOM 파일 익명화 진행
    for m in range(len(sample_path)):
        posix = Path(sample_path[m])
        idx = posix.parts.index(Path(dicom_dir[i]).name)
        subpath = hutomid[m] / Path(*posix.parts[idx + 2:])

        anony_path = save_dir / subpath
        os.makedirs(str(anony_path), exist_ok=True)

        dcm_path = str(os.path.join(base_dir,organ[i]) / posix)
        for dirpath, _, filenames in os.walk(dcm_path):
            for filename in filenames:
                anonymize_dicom_file(dcm_path, filename, anony_path, id=hutomid[m])



 75%|████████████████████████████████████████████████████████████▊                    | 3/4 [20:47<06:46, 406.59s/it]/disk1/users/da_cyh_0/miniconda3/envs/py3_11/lib/python3.11/site-packages/pydicom/valuerep.py:440: UserWarning: The value length (113) exceeds the maximum length of 64 allowed for VR UI. Invalid value for VR UI: '7,IMAGE_TYPE,SLICE_NUMBER,ECHO_NUMBER,PHASE_NUMBER,DYNAMIC_SCAN,CHEMICAL_SHIFT,DIFF_B_VALUE_NO,ASCENDING,NONE,0,0'. Please see <https://dicom.nema.org/medical/dicom/current/output/html/part05.html#table_6.2-1> for allowed values for each VR.
  warn_and_log(msg)
100%|█████████████████████████████████████████████████████████████████████████████████| 4/4 [33:52<00:00, 508.10s/it]


In [ ]:

# 5. 익명화
for k in range(len(sids)):
    sample = sample_info[sample_info['PatientID']==sids[k]]

    sample_path = sample["folder"].tolist()
    hutomid = sample['hutom_id'].tolist()    
    # 5.1 하위 폴더 기준, 모든 DICOM 파일 익명화 진행
    for m in range(len(sample_path)):
        posix = Path(sample_path[m])
        idx = posix.parts.index(Path(dicom_dir[i]).name)
        subpath = hutomid[m] / Path(*posix.parts[idx + 2:])

        anony_path = save_dir / subpath
        os.makedirs(str(anony_path), exist_ok=True)

        dcm_path = str(os.path.join(base_dir,organ[i]) / posix)
        for dirpath, _, filenames in os.walk(dcm_path):
            for filename in filenames:
                anonymize_dicom_file(dcm_path, filename, anony_path, id=hutomid[m])


    break



In [ ]:
## Start [ver.2025.08]

sample_info_list = []
for i in range(len(dicom_dir)):

    # 1. 입력 폴더 내 샘플 폴더 리스트
    folder_list = get_folder_list(dicom_dir[i])
    for j in range(len(folder_list)):

        # 1.1. 샘플 폴더 기준, DICOM 확인 및 메타 추출
        check_path = os.path.join(dicom_dir[i], folder_list[j])
        dcm_fname = check_dicom_from_folder(check_path)
        if dcm_fname:
            metadata = get_patient_info(dcm_fname)

            anony_folder = re.sub(r'[가-힣]+', '', folder_list[j])
            anony_folder = re.sub(r'\d+', replace_digits, anony_folder)
            anony_folder = re.sub(r'\s+', '', anony_folder)

            posix = PurePath(dcm_fname)
            fpath = PurePosixPath(organ[i], *posix.parts[2:-1])
            metadata['folder'] = fpath
            metadata['Center'] = center[i]
            metadata['ImportDate'] = importdate[i]
            
            sample_info_list.append(metadata)

    # 2. 샘플 메타 [반입리스트에서 중복 제거 ? default=False]
    sample_info = pd.DataFrame.from_dict(sample_info_list)
    check_dup_col = ['PatientID','PatientName','AcquisitionDate','PatientSex']
    # sample_info = sample_info.drop_duplicates(subset=check_dup_col, ignore_index=True)
    sample_info.insert(0, 'hutom_id', None)

    # 3. 중복 제거 및 HUTOM ID 부여
    meta_all = IMAGE_META[organ[i]].copy()
    ids_all = HUTOM_ID[organ[i]].copy()
    mask = ~ids_all['hutom_id'].astype(str).str.contains('FDA', case=False, na=False)
    hutom_ids = ids_all.loc[mask,'hutom_id'].dropna().unique().tolist()
    nums = [int(m.group(1)) for s in hutom_ids if (m := re.search(r'(\d+)$', s))]
    id_number = np.sort(nums)[-1] + 1

    sample_ids = sample_info['PatientID'].unique().tolist()
    for idx in range(len(sample_ids)):
        dup = meta_all[meta_all['PatientID']==sample_ids[idx]]
        if len(dup) == 0:
            hutomid = f"{organ[i]}{id_number:0{n_digits}d}"
            check_sample = sample_info[sample_info['PatientID'] == sample_ids[idx]]
            sample_info.loc[check_sample.index, "hutom_id"] = hutomid
            id_number += 1
        elif len(dup) > 0:
            hutomid = dup["hutom_id"].tolist()[0]
            check_sample = sample_info[sample_info['PatientID'] == sample_ids[idx]]
            sample_info.loc[check_sample.index, "hutom_id"] = hutomid
        else:
            print('check sample')

    # 4. Anonymous
    anonymous_dir = os.path.join(dicom_dir[i], 'ANONYMOUS')
    sample_ids = sample_info["PatientID"].unique().tolist()
    for idx in range(len(sample_ids)):
        # 4.1. 샘플 선택
        check_sample = sample_info[sample_info['PatientID'] == sample_ids[idx]]

        # 4.2. 폴더/경로, 영상 종류, HUTOM ID
        sample_path = check_sample["folder"].tolist()
        hutomid = check_sample['hutom_id'].tolist()    
        for m in range(len(sample_path)):
            # 샘플 기준, 하위 폴더 탐색
            posix = PurePath(sample_path[m])
            dcm_path = os.path.join(dicom_dir[i], posix.parts[2])

            dcm_folder_list = check_dicom_from_folder(dcm_path, return_list=True)
            for n in range(len(dcm_folder_list)):
                # 하위 폴더 기준, 익명화: Name, ID > Hutom ID
                raw_dir = dcm_folder_list[n]
                no_ko_folder = re.sub(r'[\u1100-\u11FF\u3130-\u318F\uAC00-\uD7A3]+', '', 
                                    posix.parts[2]).strip()

                save_dir = os.path.join(anonymous_dir, hutomid[m], 
                                        no_ko_folder, os.path.join(*posix.parts[3:]))

                os.makedirs(save_dir, exist_ok=True)

                for dirpath, _, filenames in os.walk(raw_dir):
                    for filename in filenames:
                        anonymize_dicom_file(raw_dir, filename, save_dir, id=hutomid[m])

    # 5. Add information > check !!!!
    ids_add = pd.DataFrame(sample_info['hutom_id'].unique().tolist(),
                        columns=['hutom_id'])
    ids_add['dicom'] = 'O'
    ids_all_add = pd.merge(ids_all, ids_add, on='hutom_id', how='outer')
    ids_all_add["dicom"] = ids_all_add["dicom_x"].fillna(ids_all_add["dicom_y"])
    ids_all_add = ids_all_add[['hutom_id','dicom','video']]
    meta_all_add = pd.concat([meta_all, sample_info], ignore_index=True)

##### OUTPUT > IMAGE_META, HUTOM_ID 업데이트 및 저장
# IMAGE_META[organ[i]]
# meta_all_add




In [ ]:
# Start [ver.2025.12]

i = 0

# 1. 입력 폴더 내 샘플 폴더 리스트
sample_info_list = []
folder_list = get_folder_list(dicom_dir[i])
for j in range(len(folder_list)):

    # 1.1. 샘플 폴더 기준, DICOM 확인 및 메타 추출
    check_path = os.path.join(dicom_dir[i], folder_list[j])
    dcm_fname = check_dicom_from_folder(check_path)
    if dcm_fname:
        metadata = get_patient_info(dcm_fname)

        anony_folder = re.sub(r'[가-힣]+', '', folder_list[j])
        anony_folder = re.sub(r'\d+', replace_digits, anony_folder)
        anony_folder = re.sub(r'\s+', '', anony_folder)

        posix = PurePath(dcm_fname)
        # fpath = PurePosixPath(organ[i], *posix.parts[2:-1])
        fpath = PurePosixPath(*posix.parts[:-1])
        metadata['folder'] = fpath
        metadata['Center'] = center[i]
        metadata['ImportDate'] = importdate[i]
        
        sample_info_list.append(metadata)

# 2. 샘플 메타 [반입리스트에서 중복 제거 ? default=False]
sample_info = pd.DataFrame.from_dict(sample_info_list)
check_dup_col = ['PatientID','PatientName','AcquisitionDate','PatientSex']
# sample_info = sample_info.drop_duplicates(subset=check_dup_col, ignore_index=True)
sample_info.insert(0, 'hutom_id', None)
sample_info['PatientID'] = sample_info['folder'].astype(str).str.split('/').str[6]

# 3. 중복 제거 및 HUTOM ID 부여
meta_all = IMAGE_META[organ[i]].copy()
ids_all = HUTOM_ID[organ[i]].copy()
mask = ~ids_all['hutom_id'].astype(str).str.contains('FDA', case=False, na=False)
hutom_ids = ids_all.loc[mask,'hutom_id'].dropna().unique().tolist()
nums = [int(m.group(1)) for s in hutom_ids if (m := re.search(r'(\d+)$', s))]
id_number = np.sort(nums)[-1] + 1

sample_ids = sample_info['PatientID'].unique().tolist()
for idx in range(len(sample_ids)):
    try:
        dup = meta_all[meta_all['PatientID']==int(sample_ids[idx])]
    except:
        dup = meta_all[meta_all['PatientID']==sample_ids[idx]]
    if len(dup) == 0:
        hutomid = f"{organ[i]}{id_number:0{n_digits}d}"
        check_sample = sample_info[sample_info['PatientID'] == sample_ids[idx]]
        sample_info.loc[check_sample.index, "hutom_id"] = hutomid
        id_number += 1
    elif len(dup) > 0:
        hutomid = dup["hutom_id"].tolist()[0]
        check_sample = sample_info[sample_info['PatientID'] == sample_ids[idx]]
        sample_info.loc[check_sample.index, "hutom_id"] = hutomid
    else:
        print('check sample')
sample_info.loc[23,'hutom_id'] = 'UGI0407'
sample_info.loc[29,'hutom_id'] = 'UGI0416'

display(sample_info)


In [ ]:
from tqdm.notebook import tqdm

anonymous_dir = os.path.join(dicom_dir[i], 'ANONYMOUS')
sample_ids = sample_info["PatientID"].unique().tolist()
for idx in tqdm(range(15, len(sample_ids))):
    # 4.1. 샘플 선택
    check_sample = sample_info[sample_info['PatientID'] == sample_ids[idx]]

    # 4.2. 폴더/경로, 영상 종류, HUTOM ID
    sample_path = check_sample["folder"].tolist()
    hutomid = check_sample['hutom_id'].tolist()    

    for m in range(len(sample_path)):
        # 샘플 기준, 하위 폴더 탐색
        posix = PurePath(sample_path[m])
        if len(posix.parts) == 8:
            dcm_path = PurePosixPath(*posix.parts[:-1])
        elif len(posix.parts) == 7:
            dcm_path = posix

        dcm_folder_list = check_dicom_from_folder(dcm_path, return_list=True)
        for n in range(len(dcm_folder_list)):
            # 하위 폴더 기준, 익명화: Name, ID > Hutom ID
            raw_dir = dcm_folder_list[n]
            save_dir = os.path.join(anonymous_dir, hutomid[m], PurePath(raw_dir).parts[-1])

            os.makedirs(save_dir, exist_ok=True)

            for dirpath, _, filenames in os.walk(raw_dir):
                for filename in filenames:
                    anonymize_dicom_file(raw_dir, filename, save_dir, id=hutomid[m])




In [ ]:
idx=15
check_sample = sample_info[sample_info['PatientID'] == sample_ids[idx]]
check_sample
# # 4.2. 폴더/경로, 영상 종류, HUTOM ID
# sample_path = check_sample["folder"].tolist()
# hutomid = check_sample['hutom_id'].tolist()    

# for m in range(len(sample_path)):
#     # 샘플 기준, 하위 폴더 탐색
#     posix = PurePath(sample_path[m])
#     dcm_path = PurePosixPath(*posix.parts[:-1])

# #     dcm_folder_list = check_dicom_from_folder(dcm_path, return_list=True)

#     break
# check_sample




In [ ]:
from pydicom.dataset import FileMetaDataset
from pydicom.uid import ExplicitVRLittleEndian, generate_uid

def ensure_valid_file_meta(ds: pydicom.dataset.Dataset):
    # 1) 본문에 섞인 0002 그룹 제거 (중요!)
    bad_0002 = [tag for tag in ds.keys() if tag.group == 0x0002]
    for tag in bad_0002:
        del ds[tag]

    # 2) file_meta를 FileMetaDataset로 강제
    fm = ds.file_meta if hasattr(ds, "file_meta") and isinstance(ds.file_meta, FileMetaDataset) else FileMetaDataset()

    # TransferSyntaxUID는 필수 (원본에 있으면 유지, 없으면 기본값 부여)
    if not getattr(fm, "TransferSyntaxUID", None):
        fm.TransferSyntaxUID = ExplicitVRLittleEndian

    # MediaStorage SOP UIDs는 있으면 맞춰줌
    if not getattr(fm, "MediaStorageSOPClassUID", None) and getattr(ds, "SOPClassUID", None):
        fm.MediaStorageSOPClassUID = ds.SOPClassUID

    if not getattr(fm, "MediaStorageSOPInstanceUID", None):
        fm.MediaStorageSOPInstanceUID = getattr(ds, "SOPInstanceUID", None) or generate_uid()

    if not getattr(fm, "ImplementationClassUID", None):
        fm.ImplementationClassUID = generate_uid()

    ds.file_meta = fm

    # preamble 보장 (없으면 추가)
    ds.preamble = getattr(ds, "preamble", b"\0" * 128)

    # TransferSyntaxUID에 맞춘 플래그 설정(기본: Explicit VR Little Endian)
    ds.is_little_endian = True
    ds.is_implicit_VR = False

    return ds

ds = ensure_valid_file_meta(ds)
ds.save_as(os.path.join(save_dir, filename))
